# Navier–Stokes Computational Laboratory · v23
**Author:** Harpreet Singh · GitHub-ready educational CFD project

This upgraded notebook combines governing equations, discretization notes, canonical solutions, a compact cavity-flow demonstration, Taylor–Green energy decay, visualizations and automated checks.

> Scientific scope: teaching/prototyping only. The compact cavity solver is not a quantitatively validated benchmark CFD implementation.

## Project architecture

```mermaid
flowchart TD
 A[Physical model and parameters] --> B[Grid and initial conditions]
 B --> C[Numerical solvers]
 C --> D[Diagnostics and verification]
 D --> E[Plots, tables and notebook report]
 E --> F[Reproducible GitHub project]
```

| Component | Responsibility |
|---|---|
| `src/navier_stokes/` | Reusable numerical routines and plots |
| `tests/` | Pytest smoke tests |
| `docs/` | Original v22 notebook |
| Main notebook | Guided experiments and visual analytics |

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from navier_stokes import __version__, poiseuille, hagen_poiseuille, stokes_first_problem, blasius, cavity_projection, taylor_green_spectral
import viz
plt.rcParams.update({"figure.figsize":(8,4),"axes.grid":True,"figure.dpi":110})
print("Navier–Stokes Lab",__version__)

## 1. Governing equations

For an incompressible Newtonian fluid:

\[
\partial_t\mathbf{u}+(\mathbf{u}\cdot\nabla)\mathbf{u}
=-\nabla p+\nu\nabla^2\mathbf{u}+\mathbf{f},\qquad
\nabla\cdot\mathbf{u}=0.
\]

The Reynolds number is \(Re=UL/\nu\). Global weak-solution existence and uniqueness are established in 2-D under standard assumptions; global smooth regularity in 3-D remains an open mathematical problem.

## 2. Numerical methods comparison

| Method | Core idea | Advantage | Limitation |
|---|---|---|---|
| Finite differences | Grid-based derivative approximations | Simple and transparent | Geometry/boundaries |
| Projection | Predict velocity, solve pressure Poisson, project | Enforces near-solenoidal flow | Pressure boundary subtleties |
| Streamfunction–vorticity | Eliminate pressure in 2-D | Compact 2-D formulation | Vorticity wall conditions |
| Fourier spectral | Derivatives in wavenumber space | High accuracy for smooth periodic fields | Periodicity/dealiasing |
| MMS | Prescribed exact solution and forcing | Code verification | Not physical validation |

## 3. Truncation error: forward vs central difference

In [ ]:
hs=np.logspace(-1,-5,12); x0=1.
ef=np.abs((np.sin(x0+hs)-np.sin(x0))/hs-np.cos(x0))
ec=np.abs((np.sin(x0+hs)-np.sin(x0-hs))/(2*hs)-np.cos(x0))
fig,ax=plt.subplots(); ax.loglog(hs,ef,'o-',label='Forward O(h)'); ax.loglog(hs,ec,'s-',label='Central O(h²)')
ax.set(xlabel='Grid spacing h',ylabel='Absolute error',title='Finite-difference truncation error'); ax.legend(); plt.show()

## 4. Canonical solutions

In [ ]:
y,ue,uf=poiseuille(); hp=hagen_poiseuille(); st=stokes_first_problem(); b=blasius()
display(pd.DataFrame([
 {'Case':'Plane Poiseuille','Metric':'max |exact-FD|','Value':np.max(np.abs(ue-uf))},
 {'Case':'Hagen–Poiseuille','Metric':'mean/max velocity','Value':hp['U_mean']/hp['U_max']},
 {'Case':'Stokes first problem','Metric':'delta99 factor','Value':st['delta99_factor']},
 {'Case':'Blasius','Metric':"f''(0)",'Value':b['fpp0']}
]))
fig,axs=plt.subplots(1,3,figsize=(13,3.5))
axs[0].plot(ue,y,label='exact'); axs[0].plot(uf,y,'--',label='FD'); axs[0].set_title('Plane Poiseuille'); axs[0].legend()
axs[1].plot(hp['u'],hp['r']); axs[1].set_title('Hagen–Poiseuille')
for i in [0,len(st['t'])//3,2*len(st['t'])//3,-1]: axs[2].plot(st['u'][i],st['y'],label=f"t={st['t'][i]:.2f}")
axs[2].set_title('Stokes first problem'); axs[2].legend(fontsize=7); plt.tight_layout(); plt.show()

## 5. Lid-driven cavity: compact projection demonstration

In [ ]:
QUICK=True
N=20 if QUICK else 40
RE_LIST=[100,400] if QUICK else [100,400,1000]
runs={}; rows=[]
for Re in RE_LIST:
 t0=time.time(); r=cavity_projection(Re=Re,N=N,Tmax=1.0 if QUICK else 4.0); runs[Re]=r
 rows.append({'Re':Re,'grid':N,'steps':r['steps'],'max divergence':r['div'],'runtime (s)':time.time()-t0})
display(pd.DataFrame(rows))

In [ ]:
r=runs[RE_LIST[-1]]
fig=viz.plot_cavity_fields(r,fields=('speed','psi','vorticity')); plt.show()
fig,ax=plt.subplots()
for Re,q in runs.items(): viz.plot_cavity_centerline(q,ax=ax,label=f'Re={Re}')
ax.legend(); plt.show()

**Interpretation:** the moving lid drives recirculation. This compact collocated implementation is for illustrating the projection workflow only; do not report its output as a Ghia benchmark match. Quantitative work needs grid/time-step refinement and a carefully verified pressure boundary treatment.

## 6. Taylor–Green vortex energy decay

In [ ]:
t,E,eps=taylor_green_spectral(N=32,nu=.02,T=5,nsteps=100)
fig,ax=plt.subplots(); ax.plot(t,E/E[0],label='computed teaching model'); ax.plot(t,np.exp(-.08*t),'--',label='analytic decay')
ax.set(xlabel='Time',ylabel='E(t)/E(0)',title='Taylor–Green energy decay'); ax.legend(); plt.show()

## 7. Automated verification

In [ ]:
checks=[
 ('Poiseuille exact/FD agreement',np.max(np.abs(ue-uf))<1e-10,float(np.max(np.abs(ue-uf)))),
 ('Blasius wall shear',abs(b['fpp0']-.3320573)<1e-5,b['fpp0']),
 ('Energy decreases',bool(np.all(np.diff(E)<0)),float(E[-1]/E[0])),
 ('Energy remains positive',bool(np.all(E>0)),float(E.min()))
]
report=pd.DataFrame([{'Check':n,'Status':'PASS' if ok else 'FAIL','Observed':v} for n,ok,v in checks])
display(report)
assert all(ok for _,ok,_ in checks)

## 8. Research-grade next steps

- Implement a staggered-grid MAC solver and verify pressure boundary conditions.
- Add CFL-based adaptive time stepping and residual histories.
- Add a proper Method of Manufactured Solutions suite and observed-order plots.
- Compare centreline velocities against tabulated Ghia benchmark data.
- Add 3-D pseudo-spectral Navier–Stokes with 2/3 dealiasing and RK time integration.
- Add CI, code coverage, saved figures and reproducible experiment metadata.

### References
Patankar, *Numerical Heat Transfer and Fluid Flow*; Gresho & Sani, *Incompressible Flow*; Ghia, Ghia & Shin (1982), lid-driven cavity benchmark; Canuto et al., *Spectral Methods*.

# 9. Extended Numerical Equations Library

This section adds commonly used equations for incompressible-flow analysis. Symbols are defined locally in each subsection; unless stated otherwise, the equations use SI units or a stated non-dimensional convention.

## 9.1 Differential operators

For a scalar field \(\phi(x,y,z,t)\) and velocity \(\mathbf u=(u,v,w)\):

**Gradient**
\[
\nabla\phi=\left(\frac{\partial\phi}{\partial x},
\frac{\partial\phi}{\partial y},\frac{\partial\phi}{\partial z}\right)
\]

**Divergence**
\[
\nabla\cdot\mathbf u=\frac{\partial u}{\partial x}
+\frac{\partial v}{\partial y}+\frac{\partial w}{\partial z}
\]

**Laplacian**
\[
\nabla^2\phi=\frac{\partial^2\phi}{\partial x^2}
+\frac{\partial^2\phi}{\partial y^2}+\frac{\partial^2\phi}{\partial z^2}
\]

**Vorticity**
\[
\boldsymbol\omega=\nabla\times\mathbf u
\]

## 9.2 Conservative and non-conservative momentum equations

For constant-density incompressible flow:

**Non-conservative form**
\[
\rho\left(\frac{\partial\mathbf u}{\partial t}
+\mathbf u\cdot\nabla\mathbf u\right)
=-\nabla p+\mu\nabla^2\mathbf u+\rho\mathbf f
\]

**Conservative form**
\[
\frac{\partial(\rho\mathbf u)}{\partial t}
+\nabla\cdot(\rho\mathbf u\otimes\mathbf u)
=-\nabla p+\nabla\cdot\boldsymbol\tau+\rho\mathbf f
\]

For a Newtonian fluid, \(\boldsymbol\tau=\mu(\nabla\mathbf u+\nabla\mathbf u^T)\) for constant viscosity and incompressible flow.

**Continuity equation**
\[
\frac{\partial\rho}{\partial t}+\nabla\cdot(\rho\mathbf u)=0,
\qquad \nabla\cdot\mathbf u=0\quad(\rho=\text{constant})
\]

## 9.3 Dimensionless equations and characteristic numbers

Using characteristic scales \(L,U,L/U\), the momentum equation becomes:

\[
\frac{\partial\mathbf u^*}{\partial t^*}
+\mathbf u^*\cdot\nabla^*\mathbf u^*
=-\nabla^*p^*+\frac{1}{Re}\nabla^{*2}\mathbf u^*
+\frac{1}{Fr^2}\mathbf e_g
\]

\[
Re=\frac{UL}{\nu},\qquad
Pe=\frac{UL}{\alpha},\qquad
Pr=\frac{\nu}{\alpha},\qquad
Sc=\frac{\nu}{D},\qquad
Ma=\frac{U}{c}
\]

where \(Re\) is Reynolds, \(Pe\) Péclet, \(Pr\) Prandtl, \(Sc\) Schmidt, and \(Ma\) Mach number. The Froude number is \(Fr=U/\sqrt{gL}\).

## 9.4 Scalar transport / advection-diffusion equation

For temperature or a passive scalar \(\phi\):

\[
\frac{\partial\phi}{\partial t}
+\mathbf u\cdot\nabla\phi
=\Gamma\nabla^2\phi+S_\phi
\]

A 1-D finite-difference form (uniform grid) is:

**Forward Euler + central diffusion + upwind advection**
\[
\phi_i^{n+1}=\phi_i^n
-\frac{u\Delta t}{\Delta x}(\phi_i^n-\phi_{i-1}^n)
+\frac{\Gamma\Delta t}{\Delta x^2}
(\phi_{i+1}^n-2\phi_i^n+\phi_{i-1}^n)
+\Delta t S_i^n
\]
for positive constant velocity \(u\).

## 9.5 Finite-difference stencils

**First derivative, forward (first order):**
\[
f'_i\approx\frac{f_{i+1}-f_i}{\Delta x}
\]

**First derivative, central (second order):**
\[
f'_i\approx\frac{f_{i+1}-f_{i-1}}{2\Delta x}
\]

**Second derivative, central (second order):**
\[
f''_i\approx\frac{f_{i+1}-2f_i+f_{i-1}}{\Delta x^2}
\]

**Fourth-order central first derivative:**
\[
f'_i\approx\frac{-f_{i+2}+8f_{i+1}-8f_{i-1}+f_{i-2}}{12\Delta x}
\]

**Fourth-order central second derivative:**
\[
f''_i\approx\frac{-f_{i+2}+16f_{i+1}-30f_i+16f_{i-1}-f_{i-2}}
{12\Delta x^2}
\]

In [ ]:
# Verify second- and fourth-order finite-difference stencils on sin(x).
def fd1_central(f, h):
    return (f[2:]-f[:-2])/(2*h)

def fd1_fourth(f, h):
    return (f[:-4]-8*f[1:-3]+8*f[3:-1]-f[4:])/(12*h)

def fd2_central(f, h):
    return (f[2:]-2*f[1:-1]+f[:-2])/h**2

def fd2_fourth(f, h):
    return (-f[4:]+16*f[3:-1]-30*f[2:-2]+16*f[1:-3]-f[:-4])/(12*h**2)

rows=[]
for n in [41,81,161,321]:
    xx=np.linspace(0,2*np.pi,n); h=xx[1]-xx[0]; ff=np.sin(xx)
    e1=np.max(np.abs(fd1_central(ff,h)-np.cos(xx[1:-1])))
    e4=np.max(np.abs(fd1_fourth(ff,h)-np.cos(xx[2:-2])))
    e2=np.max(np.abs(fd2_central(ff,h)+np.sin(xx[1:-1])))
    e24=np.max(np.abs(fd2_fourth(ff,h)+np.sin(xx[2:-2])))
    rows.append({"N":n,"h":h,"1st O(h²)":e1,"1st O(h⁴)":e4,"2nd O(h²)":e2,"2nd O(h⁴)":e24})
fd_table=pd.DataFrame(rows)
display(fd_table)
fig,ax=plt.subplots()
for col in ["1st O(h²)","1st O(h⁴)","2nd O(h²)","2nd O(h⁴)"]:
    ax.loglog(fd_table["h"],fd_table[col],"o-",label=col)
ax.invert_xaxis(); ax.set(xlabel="Grid spacing h",ylabel="Maximum error",title="Finite-difference convergence")
ax.legend(); plt.show()

## 9.6 Time integration equations

**Forward Euler**
\[
q^{n+1}=q^n+\Delta t F(q^n)
\]

**Backward Euler**
\[
q^{n+1}=q^n+\Delta t F(q^{n+1})
\]

**Crank–Nicolson**
\[
q^{n+1}=q^n+\frac{\Delta t}{2}
\left[F(q^{n+1})+F(q^n)\right]
\]

**Classical fourth-order Runge–Kutta**
\[
k_1=F(q^n),\quad k_2=F(q^n+\tfrac{\Delta t}{2}k_1)
\]
\[
k_3=F(q^n+\tfrac{\Delta t}{2}k_2),\quad
k_4=F(q^n+\Delta t k_3)
\]
\[
q^{n+1}=q^n+\frac{\Delta t}{6}(k_1+2k_2+2k_3+k_4)
\]

In [ ]:
# Compare Euler and RK4 on dq/dt=-lambda*q, whose exact solution is exp(-lambda*t).
def integrate_euler(lam,dt,T):
    q=1.; t=0.
    while t < T-1e-14:
        h=min(dt,T-t); q += h*(-lam*q); t += h
    return q

def integrate_rk4(lam,dt,T):
    q=1.; t=0.
    while t < T-1e-14:
        h=min(dt,T-t); f=lambda z:-lam*z
        k1=f(q); k2=f(q+h*k1/2); k3=f(q+h*k2/2); k4=f(q+h*k3)
        q += h*(k1+2*k2+2*k3+k4)/6; t += h
    return q

lam,T=2.,2.
time_rows=[]
for dt in [.2,.1,.05,.025]:
    exact=np.exp(-lam*T)
    time_rows.append({"dt":dt,"Euler error":abs(integrate_euler(lam,dt,T)-exact),
                      "RK4 error":abs(integrate_rk4(lam,dt,T)-exact)})
time_table=pd.DataFrame(time_rows); display(time_table)
fig,ax=plt.subplots(); ax.loglog(time_table.dt,time_table["Euler error"],"o-",label="Euler")
ax.loglog(time_table.dt,time_table["RK4 error"],"s-",label="RK4")
ax.set(xlabel="Time step",ylabel="Absolute error",title="Time-integration convergence"); ax.legend(); plt.show()

## 9.7 Pressure Poisson equation and projection method

Taking divergence of the momentum equation and imposing incompressibility gives a pressure Poisson equation. In a common incremental projection formulation:

**Predictor**
\[
\frac{\mathbf u^*-\mathbf u^n}{\Delta t}
= -\mathcal N(\mathbf u^n)+\nu\nabla^2\mathbf u^n
\]

**Pressure correction**
\[
\nabla^2 p^{n+1}=\frac{\rho}{\Delta t}\nabla\cdot\mathbf u^*
\]

**Velocity projection**
\[
\mathbf u^{n+1}=\mathbf u^*-\frac{\Delta t}{\rho}\nabla p^{n+1}
\]

The pressure boundary condition depends on the chosen splitting and physical wall condition; it must be derived and verified rather than assumed universally.

## 9.8 Vorticity transport equation

For 2-D incompressible constant-density flow:

\[
\frac{\partial\omega}{\partial t}
+u\frac{\partial\omega}{\partial x}
+v\frac{\partial\omega}{\partial y}
=\nu\left(\frac{\partial^2\omega}{\partial x^2}
+\frac{\partial^2\omega}{\partial y^2}\right)
\]

The streamfunction convention used here is:
\[
u=\frac{\partial\psi}{\partial y},\qquad
v=-\frac{\partial\psi}{\partial x},\qquad
\nabla^2\psi=-\omega.
\]

## 9.9 Boundary-layer and pipe-flow equations

**Blasius similarity equation**
\[
f'''+\frac12ff''=0,\quad f(0)=f'(0)=0,\quad f'(\infty)=1
\]

**Laminar plane Poiseuille profile**
\[
u(y)=\frac{-1}{2\mu}\frac{dp}{dx}(h^2-y^2)
\]

**Laminar circular-pipe Hagen–Poiseuille profile**
\[
u(r)=\frac{-1}{4\mu}\frac{dp}{dx}(R^2-r^2),\quad
Q=\frac{\pi R^4}{8\mu}\left(-\frac{dp}{dx}\right)
\]

**Darcy–Weisbach pressure loss**
\[
\Delta p=f_D\frac{L}{D}\frac{\rho V^2}{2}
\]
where \(f_D\) is the Darcy friction factor (not the Fanning factor).

## 9.10 Energy balance and dissipation

For a periodic incompressible flow without forcing, kinetic energy is:
\[
E(t)=\frac12\int_\Omega |\mathbf u|^2\,d\Omega
\]

For constant viscosity:
\[
\frac{dE}{dt}=-\nu\int_\Omega |\nabla\mathbf u|^2\,d\Omega\leq0
\]
under the usual periodic/no-work boundary assumptions. This provides a useful diagnostic for numerical simulations.

## 9.11 Stability criteria: CFL and diffusion limits

A common convective CFL estimate is:
\[
CFL=\Delta t\left(\frac{|u|_{max}}{\Delta x}
+\frac{|v|_{max}}{\Delta y}\right)
\]

For explicit diffusion in 2-D on a uniform grid, a common sufficient restriction is:
\[
\nu\Delta t\left(\frac{1}{\Delta x^2}+\frac{1}{\Delta y^2}\right)
\leq\frac12
\]
The precise stability region depends on the spatial discretization and time integrator.

In [ ]:
# Compute illustrative CFL and explicit diffusion time-step bounds.
def stability_estimates(umax,vmax,dx,dy,nu,cfl_target=.5):
    dt_cfl=cfl_target/(umax/dx+vmax/dy+1e-30)
    dt_diff=.5/(nu*(1/dx**2+1/dy**2))
    return {"dt_CFL":dt_cfl,"dt_diffusion_limit":dt_diff,"dt_safe_estimate":min(dt_cfl,dt_diff)}
display(pd.DataFrame([stability_estimates(1.0,.5,1/40,1/40,.01),
                      stability_estimates(2.0,1.0,1/80,1/80,.02)],
                     index=["Case A","Case B"]))

## 9.12 Integral conservation laws

**Mass conservation**
\[
\frac{d}{dt}\int_\Omega\rho\,dV
=-\int_{\partial\Omega}\rho\mathbf u\cdot\mathbf n\,dA
\]

**Momentum balance**
\[
\frac{d}{dt}\int_\Omega\rho\mathbf u\,dV
=-\int_{\partial\Omega}\rho\mathbf u(\mathbf u\cdot\mathbf n)\,dA
-\int_{\partial\Omega}p\mathbf n\,dA
+\int_{\partial\Omega}\boldsymbol\tau\mathbf n\,dA
+\int_\Omega\rho\mathbf f\,dV
\]

**Finite-volume face-flux balance**
\[
\frac{d}{dt}(\phi_P V_P)+
\sum_f (\mathbf u_f\cdot\mathbf n_f)\phi_f A_f
=\sum_f \Gamma_f(\nabla\phi)_f\cdot\mathbf n_f A_f+S_PV_P
\]

## 9.13 Error norms and observed order

For numerical solution \(q_h\) and reference \(q\):

\[
L_1=\frac1N\sum_i|q_{h,i}-q_i|,\quad
L_2=\sqrt{\frac1N\sum_i(q_{h,i}-q_i)^2},\quad
L_\infty=\max_i|q_{h,i}-q_i|
\]

If grid spacing is refined by ratio \(r=h_c/h_f\), observed order is:
\[
p_{obs}=\frac{\log(E_c/E_f)}{\log(r)}
\]

In [ ]:
# Demonstrate error norms and observed order from the Poiseuille finite-difference solution.
norm_rows=[]
for n in [11,21,41,81,161]:
    yy, exact, approx=poiseuille(n); err=approx-exact; h=yy[1]-yy[0]
    norm_rows.append({"N":n,"h":h,"L1":np.mean(np.abs(err)),"L2":np.sqrt(np.mean(err**2)),"Linf":np.max(np.abs(err))})
norm_df=pd.DataFrame(norm_rows)
norm_df["observed_order_L2"]=np.r_[np.nan,np.log(norm_df.L2.values[:-1]/norm_df.L2.values[1:])/np.log(2)]
display(norm_df)
fig,ax=plt.subplots(); ax.loglog(norm_df.h,norm_df.L2,"o-",label="L2 error")
ax.loglog(norm_df.h,norm_df.L2.iloc[0]*(norm_df.h/norm_df.h.iloc[0])**2,"--",label="O(h²) guide")
ax.invert_xaxis(); ax.set(xlabel="h",ylabel="L2 error",title="Poiseuille grid refinement"); ax.legend(); plt.show()

## 9.14 Equation reference sheet

| Topic | Equation / diagnostic |
|---|---|
| Continuity | \(\nabla\cdot\mathbf u=0\) |
| Momentum | \(\partial_t\mathbf u+\mathbf u\cdot\nabla\mathbf u=-\nabla p+\nu\nabla^2\mathbf u\) |
| Reynolds number | \(Re=UL/\nu\) |
| Advection-diffusion | \(\partial_t\phi+\mathbf u\cdot\nabla\phi=\Gamma\nabla^2\phi+S\) |
| Pressure Poisson | \(\nabla^2p=(\rho/\Delta t)\nabla\cdot\mathbf u^*\) |
| Vorticity (2-D) | \(\partial_t\omega+\mathbf u\cdot\nabla\omega=\nu\nabla^2\omega\) |
| Kinetic energy | \(E=\frac12\int|\mathbf u|^2d\Omega\) |
| Convective stability | \(CFL=\Delta t(|u|/\Delta x+|v|/\Delta y)\) |
| Error norm | \(L_2=\sqrt{N^{-1}\sum_i e_i^2}\) |
| Observed order | \(p=\log(E_c/E_f)/\log(r)\) |

**Important:** equation form, signs, boundary conditions and nondimensional scaling must be consistent with the selected coordinate system and solver formulation.

# 10. Extended 3D Visual Laboratory

This section adds four 3D views: a channel velocity-vector field, a pressure surface, helical vortex trajectories, and a structured computational mesh. The PNG previews are included in the `d3_*.png` files; the code cells below regenerate them when the notebook is run.

![3D velocity vectors](d3_01_velocity_vector_field.png)

![3D pressure surface](d3_02_pressure_surface.png)

![Vortex tube trajectories](d3_03_vortex_tube.png)

![Structured 3D mesh](d3_04_structured_mesh.png)


## 10.1 Additional governing and constitutive equations

**Material derivative**
\[
\frac{D\phi}{Dt}=\frac{\partial \phi}{\partial t}+\mathbf{u}\cdot\nabla\phi
\]

**Incompressible Newtonian momentum equation**
\[
\rho\left(\frac{\partial\mathbf{u}}{\partial t}+\mathbf{u}\cdot\nabla\mathbf{u}\right)
=-\nabla p+\mu\nabla^2\mathbf{u}+\rho\mathbf{f}
\]

**Compressible continuity**
\[
\frac{\partial\rho}{\partial t}+\nabla\cdot(\rho\mathbf{u})=0
\]

**Compressible total-energy conservation (one common form)**
\[
\frac{\partial(\rho E)}{\partial t}+\nabla\cdot[(\rho E+p)\mathbf{u}]
=\nabla\cdot(k\nabla T)+\nabla\cdot(\boldsymbol{\tau}\cdot\mathbf{u})+\rho\mathbf{f}\cdot\mathbf{u}
\]

**Newtonian viscous stress**
\[
\boldsymbol{\tau}=\mu\left[\nabla\mathbf{u}+(\nabla\mathbf{u})^T\right]
-\frac{2}{3}\mu(\nabla\cdot\mathbf{u})\mathbf{I}
\]

**Navier slip boundary condition**
\[
u_t-u_{wall,t}=b\,\frac{\partial u_t}{\partial n}
\]

**Hydrostatic pressure**
\[
p(z)=p_0+\rho g(h-z)
\]

**Bernoulli equation (steady, inviscid, along a streamline)**
\[
\frac{p}{\rho}+\frac{|\mathbf{u}|^2}{2}+gz=C
\]


## 10.2 Numerical discretization equations

**Central difference, first derivative**
\[
\left.\frac{\partial f}{\partial x}\right|_i\approx\frac{f_{i+1}-f_{i-1}}{2\Delta x}
\]

**Second-order central Laplacian in 3D**
\[
\nabla_h^2\phi_{i,j,k}=
\frac{\phi_{i+1,j,k}-2\phi_{i,j,k}+\phi_{i-1,j,k}}{\Delta x^2}
+\frac{\phi_{i,j+1,k}-2\phi_{i,j,k}+\phi_{i,j-1,k}}{\Delta y^2}
+\frac{\phi_{i,j,k+1}-2\phi_{i,j,k}+\phi_{i,j,k-1}}{\Delta z^2}
\]

**Upwind advection flux (positive face velocity)**
\[
F_{i+1/2}=u_{i+1/2}\phi_i
\]

**Finite-volume conservation**
\[
\frac{d}{dt}(\phi_P V_P)+\sum_f F_f A_f=\sum_f \Gamma_f(\nabla\phi)_f\cdot\mathbf{n}_f A_f+S_PV_P
\]

**Poisson equation and Jacobi iteration**
\[
\nabla^2p=b,\qquad
p_{i,j}^{(m+1)}=\frac{1}{2(\Delta x^{-2}+\Delta y^{-2})}
\left[\frac{p_{i+1,j}^{(m)}+p_{i-1,j}^{(m)}}{\Delta x^2}
+\frac{p_{i,j+1}^{(m)}+p_{i,j-1}^{(m)}}{\Delta y^2}-b_{i,j}\right]
\]

**Generalized-\(\alpha\) family (structural/fluid transient solvers)**
\[
M\ddot q_{n+1-\alpha_m}+C\dot q_{n+1-\alpha_f}+Kq_{n+1-\alpha_f}=f_{n+1-\alpha_f}
\]


## 10.3 Dimensionless groups and boundary models

\[
C_f=\frac{\tau_w}{\tfrac12\rho U_\infty^2},\quad
C_p=\frac{p-p_\infty}{\tfrac12\rho U_\infty^2},\quad
St=\frac{fL}{U},\quad
We=\frac{\rho U^2L}{\sigma}
\]

where \(C_f\) is skin-friction coefficient, \(C_p\) pressure coefficient, \(St\) Strouhal number, and \(We\) Weber number.

**No-penetration and no-slip:** \(\mathbf{u}=\mathbf{u}_{wall}\).

**Thermal Dirichlet:** \(T=T_w\). **Thermal Neumann:** \(-k\nabla T\cdot\mathbf{n}=q''\).

**Wall shear stress:** \(\tau_w=\mu(\partial u_t/\partial n)_{wall}\).


In [ ]:
# Regenerate the 3D figures using synthetic illustrative fields.
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 3D pressure surface
x = np.linspace(-2, 2, 80); y = np.linspace(-2, 2, 80)
X, Y = np.meshgrid(x, y)
P = np.exp(-0.8*(X**2 + Y**2))
fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(X, Y, P, cmap="viridis")
fig.colorbar(surf, ax=ax, shrink=0.65, label="scaled pressure")
ax.set(title="3D pressure field", xlabel="x", ylabel="y", zlabel="p")
plt.show()


In [ ]:
# 3D channel velocity vectors (illustrative, not a CFD solution)
x, y, z = np.meshgrid(np.linspace(0, 4, 12), np.linspace(-1, 1, 7),
                      np.linspace(-0.5, 0.5, 5), indexing="ij")
u = 1-y**2
v = 0.1*np.sin(np.pi*x/4)*np.cos(np.pi*y/2)
w = 0.08*np.cos(np.pi*x/4)*np.sin(np.pi*z)
fig = plt.figure(figsize=(9, 5)); ax = fig.add_subplot(111, projection="3d")
ax.quiver(x, y, z, u, v, w, length=0.13, normalize=True)
ax.set(title="3D channel velocity field", xlabel="x", ylabel="y", zlabel="z")
plt.show()


In [ ]:
# 3D helical vortex trajectories
t = np.linspace(0, 8*np.pi, 400)
fig = plt.figure(figsize=(8, 5)); ax = fig.add_subplot(111, projection="3d")
for phase in (0, 2*np.pi/3, 4*np.pi/3):
    ax.plot(np.cos(t+phase), np.sin(t+phase), t/(2*np.pi))
ax.set(title="Helical vortex trajectories", xlabel="x", ylabel="y", zlabel="z")
plt.show()


In [ ]:
# 3D structured mesh lines
g = np.linspace(0, 1, 6)
fig = plt.figure(figsize=(8, 5)); ax = fig.add_subplot(111, projection="3d")
for a in g:
    for b in g:
        ax.plot([0,1], [a,a], [b,b], alpha=.6)
        ax.plot([a,a], [0,1], [b,b], alpha=.6)
        ax.plot([a,a], [b,b], [0,1], alpha=.6)
ax.set(title="Structured 3D computational mesh", xlabel="x", ylabel="y", zlabel="z")
plt.show()


## 10.4 Worked numerical example: 3D Laplacian

For a uniform grid, the discrete 3D Laplacian is the sum of the second differences in each coordinate direction. The following verifies the stencil on \(\phi=x^2+y^2+z^2\), for which the exact Laplacian is \(6\) away from boundaries.


In [ ]:
# Verify the 3D Laplacian stencil on phi = x^2 + y^2 + z^2
n = 21
x = np.linspace(-1, 1, n); h = x[1] - x[0]
X, Y, Z = np.meshgrid(x, x, x, indexing="ij")
phi = X**2 + Y**2 + Z**2
lap = ((phi[2:,1:-1,1:-1] - 2*phi[1:-1,1:-1,1:-1] + phi[:-2,1:-1,1:-1]) / h**2
     + (phi[1:-1,2:,1:-1] - 2*phi[1:-1,1:-1,1:-1] + phi[1:-1,:-2,1:-1]) / h**2
     + (phi[1:-1,1:-1,2:] - 2*phi[1:-1,1:-1,1:-1] + phi[1:-1,1:-1,:-2]) / h**2)
print("Mean discrete Laplacian:", lap.mean())
print("Exact Laplacian:", 6.0)
print("Maximum absolute error:", np.max(np.abs(lap - 6.0)))
assert np.allclose(lap, 6.0, atol=1e-10)


## 10.5 Stability and solver diagnostics

**Convective CFL number**
\[
CFL=\Delta t\left(\frac{|u|_{max}}{\Delta x}+\frac{|v|_{max}}{\Delta y}+\frac{|w|_{max}}{\Delta z}\right)
\]

**Explicit diffusion restriction (conservative sufficient condition)**
\[
\Delta t\leq\frac{1}{2\nu(\Delta x^{-2}+\Delta y^{-2}+\Delta z^{-2})}
\]

**Residual norm**
\[
R_2=\sqrt{\frac{1}{N}\sum_{i=1}^N(r_i)^2},\qquad
R_\infty=\max_i|r_i|
\]

**Mass imbalance**
\[
\epsilon_m=\frac{|\dot m_{in}-\dot m_{out}|}{\max(|\dot m_{in}|,\epsilon)}
\]

The 3D figures use synthetic illustrative data to teach visualization; they are not results from a validated three-dimensional Navier–Stokes solver.


# 11. 3D Graphs, Charts, and Comparison Tables

This upgrade adds four additional 3D scientific visualizations. The plots are parameter-study illustrations and method-comparison examples; they should not be interpreted as benchmark CFD results.

![3D velocity profile](d3_05_velocity_profile_surface.png)

![3D numerical method comparison](d3_06_method_error_comparison_3d.png)

![3D energy decay](d3_07_energy_decay_surface.png)

![3D vorticity surface](d3_08_vorticity_surface.png)


## 11.1 Numerical-method comparison table

The values below are representative teaching values to explain trade-offs, not universal performance guarantees. Stability depends on the PDE, boundary conditions, grid, and time-step choice.

| Method | Formal order (space/time) | Typical strength | Main limitation |
|---|---:|---|---|
| First-order upwind | 1 / depends on time integrator | Robust for advection | Numerical diffusion |
| Second-order central | 2 / depends on time integrator | Low dissipation on smooth fields | May oscillate for advection-dominated cases |
| Fourth-order central | 4 / depends on time integrator | High accuracy on smooth structured grids | Wider stencil and boundary treatment |
| Forward Euler | — / 1 | Very simple explicit method | Strong time-step restrictions |
| RK4 | — / 4 | Accurate explicit time integration | Multiple RHS evaluations per step |
| Crank–Nicolson | — / 2 | A-stable for linear diffusion | Can show weak oscillations for large steps |
| Backward Euler | — / 1 | Strong damping and robust diffusion solve | More numerical damping |


In [ ]:
# Create an editable comparison table in the notebook
import pandas as pd

method_comparison = pd.DataFrame([
    ["First-order upwind", 1, "Advection robustness", "Numerical diffusion"],
    ["Second-order central", 2, "Good smooth-field accuracy", "Can oscillate in advection-dominated flow"],
    ["Fourth-order central", 4, "High smooth-field accuracy", "Wide stencil / boundary complexity"],
    ["Forward Euler", 1, "Simple explicit update", "Strict stability limits"],
    ["RK4", 4, "Accurate explicit time stepping", "Four RHS evaluations per step"],
    ["Crank–Nicolson", 2, "A-stable for linear diffusion", "Possible weak oscillations at large dt"],
    ["Backward Euler", 1, "Robust damping", "Numerical damping"],
], columns=["Method", "Formal order", "Strength", "Limitation"])
display(method_comparison)


## 11.2 Solver and discretization comparison

| Solver / approach | Unknowns | Typical use | Computational note |
|---|---|---|---|
| Projection method | Velocity and pressure correction | Incompressible flow | Pressure Poisson solve each step |
| SIMPLE family | Pressure-velocity coupling | Steady / pseudo-transient CFD | Iterative pressure correction |
| Finite difference | Nodal field values | Structured grids | Simple stencils |
| Finite volume | Cell averages / face fluxes | Conservation-focused CFD | Flux balance per control volume |
| Finite element | Basis-function coefficients | Complex geometries | Weak-form assembly |
| Spectral method | Modal coefficients | Smooth periodic domains | High accuracy for smooth solutions |


In [ ]:
solver_comparison = pd.DataFrame([
    ["Projection method", "Velocity + pressure correction", "Incompressible transient flow", "Pressure Poisson equation"],
    ["SIMPLE family", "Pressure-velocity coupling", "Steady / pseudo-transient flow", "Iterative correction loop"],
    ["Finite difference", "Nodal values", "Structured grids", "Stencil-based derivatives"],
    ["Finite volume", "Cell averages / face fluxes", "Conservation-focused CFD", "Control-volume flux balance"],
    ["Finite element", "Basis coefficients", "Complex geometry", "Weak-form matrix assembly"],
    ["Spectral method", "Modal coefficients", "Smooth periodic fields", "Global basis transforms"],
], columns=["Approach", "Primary unknowns", "Typical application", "Core computational step"])
display(solver_comparison)


## 11.3 3D chart: energy decay over time and viscosity

The plotted relation \(E(t,\nu)=\exp(-4\nu t)\) is a simple illustrative decay surface. It is not a general nonlinear Navier–Stokes energy solution.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

t = np.linspace(0, 3, 60)
nu = np.linspace(0.001, 0.08, 60)
T, NU = np.meshgrid(t, nu)
E = np.exp(-4*NU*T)
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(T, NU, E, cmap="plasma", edgecolor="none")
fig.colorbar(surface, ax=ax, shrink=.65, label="normalized energy")
ax.set_xlabel("time"); ax.set_ylabel("viscosity"); ax.set_zlabel("energy")
ax.set_title("3D energy-decay chart")
plt.show()


## 11.4 3D chart: vorticity distribution

An analytic illustrative scalar field is used to demonstrate 3D surface plotting and interpretation of positive/negative vorticity regions.


In [ ]:
x = np.linspace(-2, 2, 80)
y = np.linspace(-2, 2, 80)
X, Y = np.meshgrid(x, y)
r2 = X**2 + Y**2
omega = 2*np.exp(-r2)*(1-r2)
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(X, Y, omega, cmap="coolwarm", edgecolor="none")
fig.colorbar(surface, ax=ax, shrink=.65, label="scaled vorticity")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("vorticity")
ax.set_title("3D vorticity chart")
plt.show()


## 11.5 3D method-error chart

The following chart uses a deliberately synthetic error model \(e(h)=C h^p\) to illustrate how formal order changes grid-refinement trends. It is not a measured benchmark.


In [ ]:
grid_sizes = np.array([16, 32, 64, 128], dtype=float)
methods = {"Upwind-1": (0.12, 1), "Central-2": (0.08, 2), "Central-4": (0.045, 4)}
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
for j, (name, (c, p)) in enumerate(methods.items()):
    err = c*(16/grid_sizes)**p
    ax.plot(np.full_like(grid_sizes, j), grid_sizes, err, marker="o", label=name)
ax.set_xticks(range(len(methods))); ax.set_xticklabels(list(methods.keys()))
ax.set_xlabel("method"); ax.set_ylabel("grid resolution"); ax.set_zlabel("illustrative error")
ax.set_title("3D grid-refinement comparison")
ax.legend()
plt.show()


## 11.6 Quick comparison: physical models

| Model | Governing simplification | Useful regime |
|---|---|---|
| Stokes flow | Neglect inertia | Very low Reynolds number |
| Potential flow | Irrotational, inviscid approximation | Outer flow away from viscous layers |
| Boundary-layer equations | Thin viscous layer approximation | High-Reynolds-number wall flows |
| Euler equations | Inviscid compressible flow | Shock / wave-dominated idealization |
| Navier–Stokes | Viscous momentum transport | General continuum fluid flow |


# 12. CFD Simulation Laboratory — 2D Lid-Driven Cavity

This section includes a computed incompressible-flow case at \(Re=100\), not just a conceptual drawing. It uses the streamfunction–vorticity formulation on a uniform square grid. The generated results are stored in the `sim_*` files.

**Model assumptions:** constant density, Newtonian fluid, 2D laminar flow, unit-square cavity, stationary walls except the moving top lid. The numerical method uses explicit Euler time advancement, first-order upwind advection, central diffusion, and a sparse Poisson solve for streamfunction. It is an educational solver, not a validated industrial CFD code.

![CFD velocity streamlines](sim_01_cavity_velocity_streamlines.png)

![CFD streamfunction](sim_02_cavity_streamfunction.png)

![CFD vorticity](sim_03_cavity_vorticity.png)

![3D simulated velocity magnitude](sim_04_cavity_velocity_3d.png)

![CFD update history](sim_05_cavity_update_history.png)

![Centerline profiles](sim_06_cavity_centerline_profiles.png)

![Divergence diagnostic](sim_07_cavity_divergence_diagnostic.png)


## 12.1 Governing equations

**Incompressible continuity**
\[
\nabla\cdot\mathbf{u}=0
\]

**Momentum**
\[
\frac{\partial\mathbf{u}}{\partial t}+(\mathbf{u}\cdot\nabla)\mathbf{u}
=-\frac{1}{\rho}\nabla p+\nu\nabla^2\mathbf{u}
\]

For a 2D flow, define streamfunction \(\psi\) and vorticity \(\omega\):
\[
u=\frac{\partial\psi}{\partial y},\qquad
v=-\frac{\partial\psi}{\partial x},\qquad
\omega=\frac{\partial v}{\partial x}-\frac{\partial u}{\partial y}
=-\nabla^2\psi
\]

**Vorticity transport**
\[
\frac{\partial\omega}{\partial t}
+u\frac{\partial\omega}{\partial x}
+v\frac{\partial\omega}{\partial y}
=\nu\nabla^2\omega
\]

**No-slip wall vorticity (Thom-type approximation)**
\[
\omega_{wall}\approx-\frac{2(\psi_{adjacent}-\psi_{wall})}{\Delta n^2}
-\frac{2U_{wall,t}}{\Delta n}
\]
The sign of the moving-wall term depends on wall orientation and the adopted vorticity convention.


## 12.2 Further CFD equations

**Conservative momentum**
\[
\frac{\partial(\rho\mathbf{u})}{\partial t}
+\nabla\cdot(\rho\mathbf{u}\otimes\mathbf{u})
=-\nabla p+\nabla\cdot\boldsymbol{\tau}+\rho\mathbf{f}
\]

**Pressure-correction Poisson equation (projection family)**
\[
\nabla^2p'=\frac{\rho}{\Delta t}\nabla\cdot\mathbf{u}^{*}
\]

**Velocity correction**
\[
\mathbf{u}^{n+1}=\mathbf{u}^{*}-\frac{\Delta t}{\rho}\nabla p'
\]

**Finite-volume transient balance**
\[
\frac{(\rho\phi V)_P^{n+1}-(\rho\phi V)_P^n}{\Delta t}
+\sum_f \rho_f\phi_f(\mathbf{u}_f\cdot\mathbf{n}_f)A_f
=\sum_f\Gamma_f(\nabla\phi)_f\cdot\mathbf{n}_fA_f+S_\phi V_P
\]

**SIMPLE pressure correction (schematic)**
\[
a_Pp'_P=\sum_{nb}a_{nb}p'_{nb}+b_P,\qquad
\mathbf{u}_f=\mathbf{u}_f^*+d_f(p'_P-p'_{nb})
\]

**RANS Reynolds-stress closure**
\[
-\rho\overline{u_i'u_j'}=2\mu_t S_{ij}-\frac{2}{3}\rho k\delta_{ij}
\]

**Turbulent kinetic energy**
\[
\frac{\partial(\rho k)}{\partial t}+\nabla\cdot(\rho\mathbf{u}k)
=\nabla\cdot\left[\left(\mu+\frac{\mu_t}{\sigma_k}\right)\nabla k\right]
+P_k-\rho\epsilon
\]

**Wall coordinate**
\[
y^+=\frac{u_\tau y}{\nu},\qquad u_\tau=\sqrt{\frac{\tau_w}{\rho}}
\]

**Nusselt and heat-transfer relations**
\[
Nu=\frac{hL}{k},\qquad q''=-k\nabla T\cdot\mathbf{n}
\]

**Large-eddy simulation filter**
\[
\frac{\partial\bar u_i}{\partial t}
+\bar u_j\frac{\partial\bar u_i}{\partial x_j}
=-\frac{1}{\rho}\frac{\partial\bar p}{\partial x_i}
+\nu\nabla^2\bar u_i-\frac{\partial\tau_{ij}^{SGS}}{\partial x_j}
\]


## 12.3 Run the reproducible CFD solver

The solver source is also available at `src/navier_stokes/cavity_cfd.py`. Run it from the project root:

```bash
python -m pip install -r requirements.txt
python src/navier_stokes/cavity_cfd.py --n 41 --re 100 --steps 2200 --dt 0.0015
```

The next cell reruns the same case inside the notebook. It requires SciPy.


In [ ]:
from cavity_cfd import simulate_cavity
import numpy as np
import matplotlib.pyplot as plt

cfd = simulate_cavity(n=41, reynolds=100, steps=2200, dt=0.0015)
Xc, Yc = np.meshgrid(cfd["x"], cfd["y"])
fig, ax = plt.subplots(figsize=(7, 5))
levels = np.linspace(0, cfd["speed"].max(), 25)
cf = ax.contourf(Xc, Yc, cfd["speed"], levels=levels, cmap="viridis")
ax.streamplot(cfd["x"], cfd["y"], cfd["u"], cfd["v"], color="white", density=1.2)
fig.colorbar(cf, ax=ax, label="velocity magnitude")
ax.set_aspect("equal"); ax.set_title("Computed lid-driven cavity CFD")
plt.show()

print("Maximum interior discrete divergence:",
      np.max(np.abs(cfd["divergence"][1:-1, 1:-1])))
print("Maximum speed:", cfd["speed"].max())


## 12.4 CFD method comparison

| CFD approach | Pressure-velocity treatment | Typical strengths | Main trade-offs |
|---|---|---|---|
| Streamfunction–vorticity | Eliminates pressure in 2D incompressible flow | Compact for 2D teaching cases | Harder to extend to 3D |
| Projection method | Pressure Poisson correction | Clear transient incompressible workflow | Boundary conditions for pressure need care |
| SIMPLE | Iterative pressure correction | Widely used for steady incompressible CFD | Can converge slowly |
| PISO | Multiple pressure corrections per time step | Useful for transient flow | More work per time step |
| Finite volume | Face-flux conservation | Local conservation | Flux interpolation matters |
| FEM | Weak-form discretization | Complex geometry and multiphysics | Assembly and stabilization complexity |
| LES | Resolved large scales + SGS model | Unsteady turbulence structures | Expensive grid/time resolution |
| RANS | Turbulence closure models | Lower cost for mean-flow engineering | Model-form uncertainty |


## 12.5 Interpretation and limitations

- The cavity is a 2D laminar educational benchmark configuration at \(Re=100\).
- The solver does not compute a pressure field directly; pressure is eliminated through the streamfunction–vorticity formulation.
- The top corners have a lid-velocity discontinuity, which creates local numerical sensitivity.
- The update norm is a time-step change diagnostic, not a fully independent nonlinear residual.
- For quantitative validation, compare centerline velocity profiles against a published benchmark, perform grid/time-step refinement, and verify mass conservation and boundary treatment.
- Do not use this compact solver for design certification or industrial predictions.


# 13. Regenerated CFD Vorticity Analysis

The vorticity figure has been regenerated from the saved numerical simulation field in `sim_cavity_Re100_fields.npz`. Unlike a schematic illustration, these plots use the computed \(\omega\), \(u\), and \(v\) arrays from the Re=100 cavity run.

**Sign convention used by this notebook**
\[
\omega_z=\frac{\partial v}{\partial x}-\frac{\partial u}{\partial y},
\qquad
u=\frac{\partial\psi}{\partial y},\quad
v=-\frac{\partial\psi}{\partial x}
\]
Therefore,
\[
\omega_z=-\nabla^2\psi.
\]

The diverging color map is centered at zero, so positive and negative vorticity are visually distinguishable. The full computed range is retained to show the strong wall-vorticity layers.


### Regenerated 2D vorticity field

![Regenerated simulated vorticity field](sim_03_cavity_vorticity.png)

### 3D vorticity surface

![3D simulated vorticity](sim_08_cavity_vorticity_3d.png)

### Centreline vorticity profiles

![Vorticity centreline profiles](sim_09_cavity_vorticity_centerlines.png)


In [ ]:
# Recreate the vorticity plots from the stored CFD output.
from plot_vorticity import plot_vorticity
plot_vorticity("sim_cavity_Re100_fields.npz",
               output_dir=".")
print("Vorticity figures regenerated from simulation data.")
